In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_pl = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/pl_2016_2022_bn.csv")

In [3]:
df_pl = df_pl.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_pl

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,JI,FOP,PT,PR,DBDC,EUPC
0,3.82,4.0,4.30,10.58,124.0,-0.04,30.67,20.087301,17.821550,3.075475,...,61.5,0.05,-285.220000,10799.000000,0.783,0.745,79.141239,65,0.0,7.386109e+07
1,2.38,4.1,4.50,10.22,124.0,0.01,28.81,19.908145,17.566841,5.139335,...,63.0,0.07,904.510000,13375.000000,0.690,0.709,79.105836,61,0.0,1.221549e+08
2,3.35,4.1,4.80,9.62,124.0,-0.00,28.86,21.443881,17.469653,6.246071,...,64.5,-0.03,-1915.520000,-1790.000000,0.595,0.709,79.495816,62,0.0,2.523686e+08
3,3.15,4.3,4.80,9.15,124.0,-0.02,28.83,20.707679,17.818308,4.606007,...,65.3,-0.05,3853.760000,-17545.000000,0.572,0.704,76.490761,62,0.0,2.802083e+08
4,3.31,4.1,3.65,9.59,124.0,-0.18,28.57,18.839405,18.837162,-0.861202,...,65.4,-0.25,2608.050000,-16997.000000,0.570,0.707,76.831746,63,0.0,3.222559e+08
5,5.44,4.1,3.65,8.40,123.0,-0.40,28.49,21.810673,18.523151,8.471718,...,68.5,0.10,2120.610000,-2918.357143,0.542,0.641,76.516572,63,0.0,3.734915e+08
6,6.01,4.1,3.65,8.25,124.0,-2.48,29.18,22.059468,18.119129,5.712276,...,70.3,0.03,1733.861333,-2918.357143,0.570,0.616,76.516572,72,0.0,3.099679e+08


In [5]:
discretised_pl = pd.DataFrame(index=df_pl.index)

for col in df_pl.columns:
    no_unique = df_pl[col].nunique()
    
    if no_unique <= 1:
        discretised_pl[col] = 0
    else:
        try:
            discretised_pl[col] = pd.qcut(
                df_pl[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_pl[col] = df_pl[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_pl)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    1    0    0     2     0    1     2     0     1      0  ...    0    1   
1    0    0    0     2     0    2     0     0     0      1  ...    0    2   
2    1    0    1     1     0    2     1     1     0      2  ...    0    0   
3    0    1    1     0     0    1     1     1     0      0  ...    1    0   
4    0    0    0     1     0    0     0     0     2      0  ...    1    0   
5    2    0    0     0     0    0     0     2     2      2  ...    2    2   
6    2    0    0     0     0    0     2     2     1      1  ...    2    1   

   EPFRG  GGAG  JI  FOP  PT  PR  DBDC  EUPC  
0      0     2   2    2   2   2     0     0  
1      0     2   2    1   1   0     0     0  
2      0     1   1    1   2   0     0     0  
3      2     0   1    0   0   0     0     1  
4      2     0   0    1   1   1     0     2  
5      1     0   0    0   0   1     0     2  
6      1     0   0    0   0   2     0     1

In [6]:
sm = from_pandas(discretised_pl)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_pl.html")

Graphs/01_fully_connected_pl.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_pl.html")

Graphs/01_thresholded_pl.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_pl_1.html")

In [10]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)

     From     To    Weight
10    QOR     JI  2.894831
17    QOA   LBGD  2.439352
0     FDI   EBSG  2.305781
28  GDPCG     JI  1.977461
2     FDI     PR  1.839593
8     QOR   EBSG  1.732849
29    EIS  EPFRG  1.595657
16    QOA  GDPCG  1.575060
23    PGP    EII  1.396872
35     PR   VABI  1.309206
14    QOA   VABI  1.273413
19    QOA  EPFRG  1.229710
31  EPFRG   EBSG  1.202159
22    QOA     PT  1.198427
21    QOA    FOP  1.196083
9     QOR    FEM  1.154863
5     QOR   VABI  1.150591
15    QOA   GCFP  1.026735
13    QOR     PR  1.020084
33   GGAG    LFG  0.988828
24    PGP    LTU  0.963294
30    LTU   GGAG  0.803020
34     PT    LFG -0.884762
18    QOA    LTU -0.923469
32   GGAG    EII -0.939333
6     QOR   GGFC -0.953710
1     FDI     JI -0.996885
11    QOR    FOP -1.062674
3     QOR    FDI -1.094038
12    QOR     PT -1.167061
26   GCFP  EPFRG -1.198406
27   GCFP     PR -1.283102
4     QOR   EIAP -1.373602
7     QOR  GDPCG -1.700996
20    QOA     JI -1.923203
25   GCFP   EBSG -2.278554
